# KAUST Kaggle Challenge 3 - Notebook C
## Fewest-features candidate, plus the full-model candidate in one run

Two prize pools, one run:

* **fewest-features pool** (3 columns or fewer, all tied on count, highest score wins)
* **normal pool** (the 51-column no-weather model)

The minimal candidate is built around one column that does almost all the work:
`S2_z_K5`, Sensor 2 z-scored against its five nearest neighbours on each side of the
minute-floored stream. Because faults are injected into isolated single rows, a row's
neighbours are a clean estimate of the ambient level, and a faulty row sits far below it.

Everything is scored at a fixed threshold of 0.5 under **both** validation schemes.
Set `NB_FAST=1` in the environment for a two-minute smoke run before the real one.

## 0. Setup

In [ ]:
import os, sys, json, time, glob, gc, warnings, itertools
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupKFold

t_start = time.time()
SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 80)
warnings.filterwarnings("ignore", category=FutureWarning)

# FAST=1 gives a smoke run in a couple of minutes. Leave at 0 for the real run.
FAST = int(os.environ.get("NB_FAST", "0"))
ITERS      = 60 if FAST else 400
ORDERINGS  = (0,) if FAST else (0, 1, 2)
N_SEEDS    = 1 if FAST else 3
RUN_FULL   = True          # also build the 51-feature "normal pool" candidate

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
os.makedirs(WORK, exist_ok=True)

try:
    from catboost import CatBoostClassifier
    import catboost; print("catboost", catboost.__version__)
except ImportError:
    os.system(f"{sys.executable} -m pip install -q catboost")
    from catboost import CatBoostClassifier

print(f"FAST={FAST} iters={ITERS} orderings={ORDERINGS} seeds={N_SEEDS}")
print("work dir:", WORK)

In [ ]:
def find_files():
    for root in ["/kaggle/input", ".", "..", "/mnt/user-data/uploads"]:
        if not os.path.isdir(root):
            continue
        found = {}
        for p in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
            b = os.path.basename(p).lower()
            if b.startswith("train") and "train" not in found: found["train"] = p
            elif b.startswith("test") and "test" not in found: found["test"] = p
            elif "sample" in b and "sample" not in found: found["sample"] = p
        if "train" in found and "test" in found:
            return found
    raise FileNotFoundError("could not locate train/test under any known root")

FILES = find_files()
for k, v in FILES.items():
    print(f"{k:7s} -> {v} ({os.path.getsize(v)/1e6:.1f} MB)")

def parse_datetime(s):
    s = s.astype(str)
    for fmt in ("%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%Y-%m-%d %H:%M"):
        out = pd.to_datetime(s, format=fmt, errors="coerce")
        if out.notna().all():
            return out, fmt
    out = pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=False)
    if out.isna().any():
        raise ValueError("unparseable timestamps")
    return out, "mixed"

train = pd.read_csv(FILES["train"]); test = pd.read_csv(FILES["test"])
sample = pd.read_csv(FILES["sample"]) if "sample" in FILES else None
train["dt"], f1_ = parse_datetime(train["Datetime"])
test["dt"],  f2_ = parse_datetime(test["Datetime"])
S1, S2, TARGET, EPS = "Sensor1_PM2.5", "Sensor2_PM2.5", "Offset_fault", 1e-3

assert test["dt"].between(pd.Timestamp("2021-11-30"), pd.Timestamp("2022-02-01")).all()
assert train["dt"].max() < test["dt"].min()
print(f"train fmt {f1_} {train.shape} | test fmt {f2_} {test.shape}")
print("train", train['dt'].min(), "->", train['dt'].max())
print("test ", test['dt'].min(), "->", test['dt'].max())
print("positive rate", round(train[TARGET].mean(), 5))

## 1. Feature construction

In [ ]:
# ---------------------------------------------------------------------------
# Feature construction
#
# Both tables are floored to the minute first. Test timestamps carry no
# seconds, so ordering train more precisely than test would measure a
# precision we never have at inference. Within-minute ties are broken by a
# seeded random draw, applied identically to train and test.
#
# The minimal feature set is deliberately tiny:
#   S2_z_K5  - Sensor 2 z-scored against its 5 nearest neighbours each side
#   S1_z_K5  - the same for Sensor 1
#   sym_rel  - symmetric relative difference between the two sensors
# ---------------------------------------------------------------------------
K_MAIN = 5

def neighbour_frame(base, K):
    cols = {}
    for j in range(1, K + 1):
        cols[f"m{j}"] = base.shift(j)
        cols[f"p{j}"] = base.shift(-j)
    return pd.DataFrame(cols)

def build_stream(order_seed=0):
    a = train.copy(); b = test.copy()
    a["ID"] = -1; a["is_test"] = 0
    b["is_test"] = 1; b[TARGET] = np.nan
    full = pd.concat([a, b], ignore_index=True, sort=False)
    full["tmin"] = full["dt"].dt.floor("min")
    full["_tie"] = np.random.RandomState(order_seed).rand(len(full))
    return full.sort_values(["tmin", "_tie"]).reset_index(drop=True)

def build_minimal(full, Ks=(K_MAIN,)):
    """Small, interpretable columns. Returns df plus the canonical trio."""
    f = pd.DataFrame(index=full.index)
    diff = full[S1] - full[S2]
    f["sym_rel"] = 2 * diff / (full[S1].abs() + full[S2].abs() + EPS)
    f["rel_diff"] = diff / (full[S2].abs() + EPS)
    for K in Ks:
        for col, nm in ((S2, "S2"), (S1, "S1")):
            N = neighbour_frame(full[col], K)
            f[f"{nm}_z_K{K}"] = (full[col] - N.mean(axis=1)) / (N.std(axis=1) + EPS)
    return f.replace([np.inf, -np.inf], np.nan).astype(np.float32)

def build_full(full, K=3):
    """The 51-column no-weather set that won the earlier notebooks."""
    f = pd.DataFrame(index=full.index)
    f["S1"] = full[S1]; f["S2"] = full[S2]
    diff = full[S1] - full[S2]
    f["diff"] = diff; f["abs_diff"] = diff.abs()
    f["ratio"] = full[S1] / (full[S2].abs() + EPS)
    f["rel_diff"] = diff / (full[S2].abs() + EPS)
    f["sym_rel"] = 2 * diff / (full[S1].abs() + full[S2].abs() + EPS)
    f["log_ratio"] = np.log(full[S1].clip(lower=.01)) - np.log(full[S2].clip(lower=.01))
    f["level"] = (full[S1] + full[S2]) / 2
    f["resid_line"] = full[S1] - (1.0039 * full[S2] + 0.7551)
    ratio = f["ratio"]
    for col, nm in ((S1, "S1"), (S2, "S2"), (None, "ratio")):
        base = ratio if col is None else full[col]
        sides = {}
        for j in range(1, K + 1):
            sides[f"m{j}"] = base.shift(j); sides[f"p{j}"] = base.shift(-j)
            if col is not None:
                f[f"{nm}_lag{j}"] = sides[f"m{j}"]; f[f"{nm}_lead{j}"] = sides[f"p{j}"]
        N = pd.DataFrame(sides)
        med, mean, sd = N.median(axis=1), N.mean(axis=1), N.std(axis=1)
        f[f"{nm}_nb_med"] = med; f[f"{nm}_nb_std"] = sd
        f[f"{nm}_nb_dev"] = base - med
        f[f"{nm}_nb_ratio"] = base / (med.abs() + EPS)
        f[f"{nm}_nb_z"] = (base - mean) / (sd + EPS)
    f["nb_ratio_gap"] = f["S1_nb_ratio"] / (f["S2_nb_ratio"].abs() + EPS)
    f["nb_dev_gap"] = f["S1_nb_dev"] - f["S2_nb_dev"]
    f["gap_prev_s"] = (full["tmin"] - full["tmin"].shift(1)).dt.total_seconds()
    f["gap_next_s"] = (full["tmin"].shift(-1) - full["tmin"]).dt.total_seconds()
    g = full.set_index("tmin"); rv = ratio.values
    for w in ("30min", "360min"):
        for col, nm in ((S1, "S1"), (S2, "S2")):
            m = g[col].rolling(w, center=True, closed="both").median().values
            f[f"{nm}_roll_ratio_{w}"] = full[col].values / (np.abs(m) + EPS)
            f[f"{nm}_roll_dev_{w}"] = full[col].values - m
        f[f"ratio_roll_dev_{w}"] = rv - (g[S1] / (g[S2].abs() + EPS)).rolling(
            w, center=True, closed="both").median().values
    return f.replace([np.inf, -np.inf], np.nan).astype(np.float32)

t0 = time.time()
FULL0 = build_stream(0)
MIN0 = build_minimal(FULL0, Ks=(3, 4, 5, 6))
is_tr = (FULL0["is_test"] == 0).values
Xmin  = MIN0[is_tr].reset_index(drop=True)
y_all = FULL0.loc[is_tr, TARGET].astype(int).values
dt_all = FULL0.loc[is_tr, "dt"].reset_index(drop=True)
print(f"minimal matrix {MIN0.shape} in {time.time()-t0:.0f}s")
print("columns:", list(MIN0.columns))

TRIO = ["S2_z_K5", "S1_z_K5", "sym_rel"]
SOLO = ["S2_z_K5"]

## 2. Validation framework

In [ ]:
# ---------------------------------------------------------------------------
# Two validation schemes, applied to every candidate.
#   forward  - train on the past, validate on the days that follow
#   grouped  - GroupKFold on calendar date
# Threshold is fixed at 0.5: earlier runs showed tuned thresholds swung
# wildly between folds and never beat 0.5 on a true future holdout.
# ---------------------------------------------------------------------------
CB = dict(iterations=ITERS, learning_rate=0.1, depth=6, l2_leaf_reg=3.0,
          loss_function="Logloss", random_seed=SEED, thread_count=-1,
          verbose=0, allow_writing_files=False)

FWD_FOLDS = [("2021-11-09", "2021-11-16"),
             ("2021-11-16", "2021-11-23"),
             ("2021-11-23", "2021-12-01")]
HOLDOUT_START = "2021-11-24"
LOG = []

def _fit(Xtr, ytr, Xva, params=None):
    m = CatBoostClassifier(**(params or CB)); m.fit(Xtr, ytr)
    return m, m.predict_proba(Xva)[:, 1]

def forward_eval(X, cols, label, params=None, verbose=True):
    sc, det = [], []
    for a, b in FWD_FOLDS:
        itr = (dt_all < pd.Timestamp(a)).values
        iva = ((dt_all >= pd.Timestamp(a)) & (dt_all < pd.Timestamp(b))).values
        _, p = _fit(X.loc[itr, cols], y_all[itr], X.loc[iva, cols], params)
        pred = (p >= 0.5).astype(int)
        pr, rc, f, _ = precision_recall_fscore_support(y_all[iva], pred, average="binary", zero_division=0)
        sc.append(f); det.append((pr, rc, pred.mean()))
    itr = (dt_all < pd.Timestamp(HOLDOUT_START)).values
    _, ph = _fit(X.loc[itr, cols], y_all[itr], X.loc[~itr, cols], params)
    hold = f1_score(y_all[~itr], (ph >= 0.5).astype(int))
    row = dict(candidate=label, n_features=len(cols), scheme="forward",
               f1_mean=float(np.mean(sc)), f1_std=float(np.std(sc)),
               f1_last=sc[-1], holdout_f1=float(hold),
               precision=det[-1][0], recall=det[-1][1], pos_rate=det[-1][2])
    LOG.append(row)
    if verbose:
        print(f"  forward {label:34s} {np.mean(sc):.5f} +/- {np.std(sc):.5f} | holdout {hold:.5f}")
    return row

def grouped_eval(X, cols, label, n_splits=5, params=None, verbose=True):
    grp = dt_all.dt.date.astype(str).values
    oof = np.zeros(len(X)); sc = []
    for itr, iva in GroupKFold(n_splits=n_splits).split(X, y_all, grp):
        _, p = _fit(X.loc[itr, cols], y_all[itr], X.loc[iva, cols], params)
        oof[iva] = p
        sc.append(f1_score(y_all[iva], (p >= 0.5).astype(int)))
    pred = (oof >= 0.5).astype(int)
    pr, rc, f, _ = precision_recall_fscore_support(y_all, pred, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_all, pred, labels=[0, 1]).ravel()
    row = dict(candidate=label, n_features=len(cols), scheme="grouped",
               f1_mean=float(np.mean(sc)), f1_std=float(np.std(sc)),
               oof_f1=float(f), precision=float(pr), recall=float(rc),
               pos_rate=float(pred.mean()), auc=float(roc_auc_score(y_all, oof)),
               tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))
    LOG.append(row)
    if verbose:
        print(f"  grouped {label:34s} {np.mean(sc):.5f} +/- {np.std(sc):.5f} | OOF {f:.5f} "
              f"P {pr:.4f} R {rc:.4f}")
    return row

## 3. Neighbourhood width
K is the only real hyperparameter of the anchor feature, so it gets measured.

In [ ]:
# ---- confirm the neighbourhood width -------------------------------------
print("neighbourhood width sweep for the anchor feature")
for K in (3, 4, 5, 6):
    forward_eval(Xmin, [f"S2_z_K{K}"], f"S2_z_K{K} (solo)")
print()
for K in (3, 4, 5, 6):
    grouped_eval(Xmin, [f"S2_z_K{K}"], f"S2_z_K{K} (solo)")

## 4. Tier-eligible candidates

In [ ]:
# ---- the tier-eligible candidates (3 features or fewer) -------------------
CANDIDATES = {
    "TRIO S2z+S1z+symrel": TRIO,
    "SOLO S2z":            SOLO,
    "PAIR S2z+S1z":        ["S2_z_K5", "S1_z_K5"],
    "PAIR S2z+symrel":     ["S2_z_K5", "sym_rel"],
    "TRIO S2z+S1z+reldif": ["S2_z_K5", "S1_z_K5", "rel_diff"],
    "TRIO multiscale":     ["S2_z_K5", "S2_z_K3", "S1_z_K5"],
}
print("forward-chaining")
for name, cols in CANDIDATES.items():
    forward_eval(Xmin, cols, name)
print("\ndate-grouped")
for name, cols in CANDIDATES.items():
    grouped_eval(Xmin, cols, name)

res = pd.DataFrame(LOG)
piv = res.pivot_table(index=["candidate", "n_features"], columns="scheme",
                      values="f1_mean").round(5)
print("\n" + piv.to_string())

## 5. Ordering sensitivity

In [ ]:
# ---- how much does the arbitrary within-minute ordering move a 3-col model?
# The minimal model leans hard on row order, so this matters more here than
# it did for the 51-column blend.
print("ordering sensitivity of the trio")
ord_scores = []
for s in ORDERINGS:
    fs = build_stream(s)
    ms = build_minimal(fs, Ks=(3, 4, 5, 6))
    mtr = (fs["is_test"] == 0).values
    Xs = ms[mtr].reset_index(drop=True)
    ys = fs.loc[mtr, TARGET].astype(int).values
    dts = fs.loc[mtr, "dt"].reset_index(drop=True)
    itr = (dts < pd.Timestamp(HOLDOUT_START)).values
    m = CatBoostClassifier(**CB); m.fit(Xs.loc[itr, TRIO], ys[itr])
    p = m.predict_proba(Xs.loc[~itr, TRIO])[:, 1]
    f = f1_score(ys[~itr], (p >= 0.5).astype(int))
    ord_scores.append(f)
    print(f"  ordering seed {s}: holdout F1 {f:.5f}")
    del fs, ms, Xs; gc.collect()
print(f"  spread {np.std(ord_scores):.5f} (min {min(ord_scores):.5f}, max {max(ord_scores):.5f})")
LOG.append(dict(candidate="ordering_spread_trio", n_features=3, scheme="ordering",
                f1_mean=float(np.mean(ord_scores)), f1_std=float(np.std(ord_scores))))

## 6. Full-model reference

In [ ]:
# ---- reference: the 51-column no-weather model (normal pool) --------------
if RUN_FULL:
    t0 = time.time()
    FULLF0 = build_full(FULL0, K=3)
    Xfull = FULLF0[is_tr].reset_index(drop=True)
    print(f"full matrix {FULLF0.shape} in {time.time()-t0:.0f}s")
    forward_eval(Xfull, list(Xfull.columns), "FULL 51col no-weather")
    grouped_eval(Xfull, list(Xfull.columns), "FULL 51col no-weather")
    print("\nThis is the cost of the fewest-features prize, measured on the same folds.")

## 7. Final training and submissions

In [ ]:
# ---- final models on all training data + submissions ----------------------
def submit(build_fn, cols, tag, seeds=N_SEEDS, orderings=ORDERINGS, params=None):
    t0 = time.time(); acc = None; ids_ref = None
    for os_ in orderings:
        fs = build_stream(os_)
        fm = build_fn(fs)
        mtr = (fs["is_test"] == 0).values
        Xa = fm[mtr].reset_index(drop=True); ya = fs.loc[mtr, TARGET].astype(int).values
        Xt = fm[~mtr].reset_index(drop=True); ids = fs.loc[~mtr, "ID"].astype(int).values
        o = np.argsort(ids); ids_ref = ids[o]
        pp = np.zeros(len(Xt))
        for s in range(seeds):
            p = dict(params or CB); p["random_seed"] = SEED + s
            m = CatBoostClassifier(**p); m.fit(Xa[cols], ya)
            pp += m.predict_proba(Xt[cols])[:, 1]
        pp /= seeds
        acc = pp[o] if acc is None else acc + pp[o]
        del fs, fm, Xa, Xt; gc.collect()
    acc /= len(orderings)
    sub = pd.DataFrame({"ID": ids_ref, TARGET: (acc >= 0.5).astype(int)})
    if sample is not None:
        assert set(sub["ID"]) == set(sample["ID"]), "ID mismatch vs sample_submission"
        sub = sub.set_index("ID").loc[sample["ID"]].reset_index()
    path = os.path.join(WORK, f"submission_{tag}.csv")
    sub.to_csv(path, index=False)
    np.save(os.path.join(WORK, f"proba_{tag}.npy"), acc)
    print(f"{tag:28s} n_feat={len(cols):2d} pos_rate={sub[TARGET].mean():.5f} "
          f"({time.time()-t0:.0f}s) -> {os.path.basename(path)}")
    return sub

mini_builder = lambda fs: build_minimal(fs, Ks=(3, 4, 5, 6))
subs = {}
subs["C3_trio"] = submit(mini_builder, TRIO, "C3_trio_3feat")
subs["C3_solo"] = submit(mini_builder, SOLO, "C3_solo_1feat")
if RUN_FULL:
    subs["C3_full"] = submit(lambda fs: build_full(fs, K=3), list(Xfull.columns), "C3_full_51feat")

print("\npairwise agreement")
ks = list(subs)
for i in range(len(ks)):
    for j in range(i + 1, len(ks)):
        a = subs[ks[i]][TARGET].values; b = subs[ks[j]][TARGET].values
        print(f"  {ks[i]:10s} vs {ks[j]:10s}: {(a == b).mean():.5f}")

pd.DataFrame(LOG).to_csv(os.path.join(WORK, "notebookC_metrics.csv"), index=False)
print("\nfiles in", WORK)
for f_ in sorted(os.listdir(WORK)):
    print("  ", f_)
print(f"\ntotal runtime {(time.time()-t_start)/60:.1f} min")
print("Nothing submitted. Send me notebookC_metrics.csv and the pos_rates.")

## What to send back

* `notebookC_metrics.csv`
* the printed positive rates and the pairwise agreement block
* the ordering-spread line from section 5

Decision rule I'd suggest once the numbers are in: if the trio and the solo are within
one ordering-spread of each other, take the trio, because its extra two columns cover
the case where Sensor 2's own neighbourhood is disturbed, which is the failure mode most
likely to differ between November and January. If the solo is clearly ahead, take the solo.